# LIMINA -- 02. Preprocessing, Label, dan Normalisasi

Mengubah enam tabel mentah (di `data/raw/`, hasil notebook 01) menjadi:

- `data/panel.csv` -- data latih, sampel berimbang 1 positif : 3 negatif
- `data/snapshot_<tanggal>.csv` -- enam potret evaluasi, seluruh emiten, proporsi kejadian apa adanya
- `data/jendela_latih.json` -- cutoff dan tanggal potret siklus ini, dibaca ulang oleh notebook 03 dan 04 supaya seluruh siklus memakai jendela waktu yang sama

Standardisasi (StandardScaler) **tidak difinalkan di sini** -- scaler produksi
dilatih di notebook 03, hanya pada data latih, karena itulah aturan yang
mengikat (lihat `docs/rancangan/AMBA-struktur-model-dan-algoritma.md` bagian
3.4: scaler yang dilatih di atas data yang sudah tercampur potret uji adalah
kebocoran). Bagian akhir notebook ini hanya pratinjau visual, bukan artefak
yang dipakai model.

In [1]:
import sys
from pathlib import Path


def _cari_root(mulai: Path) -> Path:
    for kandidat in [mulai, *mulai.parents]:
        if (kandidat / "limina" / "__init__.py").exists():
            return kandidat
    raise RuntimeError(
        "Tidak menemukan folder 'limina/' di direktori ini atau induknya. "
        "Jalankan notebook dari dalam folder proyek LIMINA."
    )


ROOT = _cari_root(Path.cwd())
sys.path.insert(0, str(ROOT))

import json
from datetime import datetime, timezone

import pandas as pd
from sklearn.preprocessing import StandardScaler

from limina import config, contracts, labels, raw_ingest, splits

df_qf = pd.read_csv(config.RAW_DIR / f"{config.TABEL_QUARTERLY_FINANCIALS}.csv")
df_dt = pd.read_csv(config.RAW_DIR / f"{config.TABEL_DAILY_TRANSACTION}.csv")
df_dfu = pd.read_csv(config.RAW_DIR / f"{config.TABEL_DAILY_FULL_UNIVERSE_CLOSE}.csv")
df_ff = pd.read_csv(config.RAW_DIR / f"{config.TABEL_FREE_FLOAT_SNAPSHOT}.csv")
df_co = pd.read_csv(config.RAW_DIR / f"{config.TABEL_COMPANY_OVERVIEW}.csv")
df_suspensi_raw = pd.read_csv(config.RAW_DIR / f"{config.TABEL_SUSPENSI}.csv")

print(f"quarterly_financials : {len(df_qf):>8,} baris")
print(f"harga (2 tabel)       : {len(df_dt) + len(df_dfu):>8,} baris")
print(f"free_float_snapshot   : {len(df_ff):>8,} baris")
print(f"company_overview      : {len(df_co):>8,} baris")
print(f"stock_suspensions     : {len(df_suspensi_raw):>8,} baris")

quarterly_financials :       48 baris
harga (2 tabel)       :    2,728 baris
free_float_snapshot   :    1,922 baris
company_overview      :       12 baris
stock_suspensions     :      588 baris


## 1. Klasifikasi alasan suspensi

In [2]:
taksonomi = labels.muat_taksonomi()
df_suspensi_raw["event_category"] = df_suspensi_raw["reason"].apply(
    lambda a: labels.klasifikasi_alasan(a, taksonomi)
)

print(df_suspensi_raw["event_category"].value_counts(dropna=False))

tak_terklasifikasi = df_suspensi_raw[df_suspensi_raw["event_category"].isna()]
if len(tak_terklasifikasi):
    print(f"\n{len(tak_terklasifikasi)} baris tidak cocok kata kunci manapun, contoh:")
    print(tak_terklasifikasi[["symbol", "reason"]].head(10).to_string(index=False))
    print(
        "\nTinjau baris di atas secara manual. Kalau ternyata pola sah yang "
        "belum tercakup, tambahkan kata kuncinya ke "
        "data/labels/taksonomi_alasan_suspensi.json."
    )

df_suspensi_c = df_suspensi_raw[df_suspensi_raw["event_category"] == labels.KATEGORI_LABEL_POSITIF].copy()
print(f"\n{len(df_suspensi_c)} peristiwa kategori C (label positif) dari {len(df_suspensi_raw)} total")

event_category
B      464
C       64
NaN     56
A        4
Name: count, dtype: int64

56 baris tidak cocok kata kunci manapun, contoh:
 symbol                    reason
TOYS.JK Suspend more than 6 month
WMPP.JK Suspend more than 6 month
KAYU.JK Suspend more than 6 month
BOSS.JK Suspend more than 6 month
DEAL.JK Suspend more than 6 month
ETWA.JK Suspend more than 6 month
TECH.JK Suspend more than 6 month
TOPS.JK Suspend more than 6 month
BIKA.JK Suspend more than 6 month
GLOB.JK Suspend more than 6 month

Tinjau baris di atas secara manual. Kalau ternyata pola sah yang belum tercakup, tambahkan kata kuncinya ke data/labels/taksonomi_alasan_suspensi.json.

64 peristiwa kategori C (label positif) dari 588 total


## 2. Peta sektor, papan pencatatan, dan cakupan emiten

In [3]:
peta_sektor = raw_ingest.bangun_peta_sektor(df_ff, df_co)
peta_board = raw_ingest.bangun_peta_board(df_co)
df_harga = raw_ingest.gabungkan_harga(df_dt, df_dfu)

symbols_universe = sorted(set(df_qf["symbol"]) | set(df_co["symbol"]) | set(df_harga["symbol"]))
print(f"Cakupan emiten: {len(symbols_universe)}")
print(f"Emiten dengan board diketahui (dari company_overview): {len(peta_board)}")
print(f"Emiten dengan sector diketahui: {len(peta_sektor)}")

Cakupan emiten: 962
Emiten dengan board diketahui (dari company_overview): 12
Emiten dengan sector diketahui: 12


## 3. Diagnosa cakupan data mentah

Sebelum membangun panel/potret, periksa dulu apakah `symbols_universe` benar-benar tumpang tindih dengan symbol di `quarterly_financials`/tabel harga, dan seberapa jauh riwayat `report_date`/tanggal harga yang tersedia. Kalau tumpang tindihnya jauh di bawah 100% atau rentang tanggalnya tidak mencapai jendela yang dibutuhkan bagian berikutnya, `data_complete` akan bernilai 0 untuk sebagian besar atau semua baris pada langkah selanjutnya -- inilah tempat pertama untuk memeriksanya, bukan menunggu notebook 03 gagal dengan galat yang membingungkan.

In [4]:
diagnosa = raw_ingest.diagnosa_cakupan_mentah(symbols_universe, df_qf, df_harga, df_suspensi_c)
for k, v in diagnosa.items():
    print(f"{k}: {v}")

if diagnosa["tumpang_tindih_quarterly_financials_persen"] < 90 or diagnosa["tumpang_tindih_harga_persen"] < 90:
    print(
        "\nPERINGATAN: tumpang tindih symbol di bawah 90 persen. Bandingkan "
        "contoh_symbol_universe dengan contoh_symbol_quarterly_financials / "
        "contoh_symbol_harga di atas -- kemungkinan besar formatnya berbeda "
        "(mis. akhiran '.JK' ada di satu tabel, tidak ada di tabel lain). "
        "Seragamkan formatnya sebelum lanjut, atau data_complete akan 0 untuk "
        "hampir semua baris pada langkah berikutnya."
    )

if diagnosa.get("jumlah_symbol_kategori_c_siap_dilatih", 0) == 0 and diagnosa.get("jumlah_symbol_kategori_c", 0) > 0:
    print(
        f"\nPERINGATAN PENTING: dari {diagnosa['jumlah_symbol_kategori_c']} symbol dengan "
        f"peristiwa kategori C, {diagnosa['jumlah_symbol_kategori_c_dengan_quarterly_financials']} "
        f"punya baris di quarterly_financials, tapi TIDAK SATU PUN yang riwayat harganya "
        f"(daily_transaction) benar-benar mencakup tanggal peristiwanya. Panel latih akan "
        f"kosong (0 baris lengkap) pada langkah berikutnya -- ini BUKAN bug, tapi cakupan "
        f"data yang belum cukup: quarterly_financials/daily_transaction Anda saat ini hanya "
        f"mencakup sebagian kecil emiten, dan riwayatnya belum menjangkau tanggal peristiwa "
        f"yang dibutuhkan. Perluas cakupan KEDUA tabel itu (bukan cuma salah satu), idealnya "
        f"ke emiten yang benar-benar pernah kena kategori C -- lihat "
        f"symbol_kategori_c_belum_punya_quarterly_financials_contoh di atas untuk daftar "
        f"prioritasnya."
    )

jumlah_symbols_universe: 962
tumpang_tindih_quarterly_financials_persen: 1.2
tumpang_tindih_harga_persen: 100.0
contoh_symbol_universe: ['AADI.JK', 'AALI.JK', 'ABBA.JK', 'ABDA.JK', 'ABMM.JK']
contoh_symbol_quarterly_financials: ['AADI.JK', 'AMMN.JK', 'ASII.JK', 'BBCA.JK', 'BELI.JK']
contoh_symbol_harga: ['AADI.JK', 'AALI.JK', 'ABBA.JK', 'ABDA.JK', 'ABMM.JK']
report_date_min: 2024-12-31
report_date_max: 2026-06-30
harga_date_min: 2026-06-15
harga_date_max: 2026-09-11
jumlah_symbol_kategori_c: 46
jumlah_symbol_kategori_c_dengan_quarterly_financials: 1
symbol_kategori_c_dengan_quarterly_financials: ['MGLV.JK']
jumlah_symbol_kategori_c_siap_dilatih: 0
symbol_kategori_c_belum_punya_quarterly_financials_contoh: ['AKKU.JK', 'ALTO.JK', 'AMMS.JK', 'ASLI.JK', 'BCIC.JK', 'BEBS.JK', 'BIMA.JK', 'COAL.JK', 'DADA.JK', 'DART.JK', 'DPNS.JK', 'FASW.JK', 'FIMP.JK', 'FISH.JK', 'GGRP.JK']

PERINGATAN: tumpang tindih symbol di bawah 90 persen. Bandingkan contoh_symbol_universe dengan contoh_symbol_quarterly

## 4. Jendela latih/evaluasi bergulir

Dihitung ULANG setiap kali cell ini jalan, relatif ke hari ini -- bukan
tanggal tetap. Ini yang membuat siklus latih besok otomatis bergeser maju
tanpa menyunting kode apa pun (lihat `limina/splits.py`).

In [5]:
cutoff_latih, tanggal_potret = splits.hitung_jendela_bergulir()
batas_matang = splits.batas_label_matang()

print(f"Dijalankan pada     : {datetime.now(timezone.utc).isoformat()}")
print(f"Batas label matang  : {batas_matang.date()}")
print(f"Cutoff latih        : {cutoff_latih.date()}")
print(f"Tanggal potret (6)  : {tanggal_potret}")

config.DATA_DIR.mkdir(parents=True, exist_ok=True)
with open(config.PATH_JENDELA, "w", encoding="utf-8") as f:
    json.dump(
        {
            "dihitung_pada": datetime.now(timezone.utc).isoformat(),
            "cutoff_latih": cutoff_latih.strftime("%Y-%m-%d"),
            "tanggal_potret": tanggal_potret,
        },
        f,
        indent=2,
    )

Dijalankan pada     : 2026-09-15T10:02:03.063276+00:00
Batas label matang  : 2026-06-17
Cutoff latih        : 2026-01-18
Tanggal potret (6)  : ['2026-01-18', '2026-02-17', '2026-03-19', '2026-04-18', '2026-05-18', '2026-06-17']


## 5. Bangun enam potret evaluasi (label sungguhan)

In [6]:
snapshot_dict = {}
for tanggal in tanggal_potret:
    snap = raw_ingest.bangun_snapshot_pasar(
        tanggal, symbols_universe, df_qf, df_harga, peta_sektor,
        peta_board=peta_board, df_suspensi_c=df_suspensi_c, taksonomi=taksonomi,
    )
    contracts.validate_panel(snap, ketat=True)  # melempar error kalau kontrak dilanggar
    snap.to_csv(config.path_snapshot(tanggal), index=False)
    snapshot_dict[tanggal] = snap
    print(
        f"potret {tanggal}: {len(snap):>5} emiten, "
        f"{int(snap['is_event_90d'].sum())} peristiwa dalam 90 hari, "
        f"{int((snap['data_complete'] == 0).sum())} tidak lengkap"
    )

potret 2026-01-18:   962 emiten, 4 peristiwa dalam 90 hari, 962 tidak lengkap
potret 2026-02-17:   962 emiten, 3 peristiwa dalam 90 hari, 962 tidak lengkap
potret 2026-03-19:   962 emiten, 3 peristiwa dalam 90 hari, 962 tidak lengkap
potret 2026-04-18:   962 emiten, 1 peristiwa dalam 90 hari, 962 tidak lengkap
potret 2026-05-18:   962 emiten, 2 peristiwa dalam 90 hari, 962 tidak lengkap
potret 2026-06-17:   962 emiten, 3 peristiwa dalam 90 hari, 962 tidak lengkap


## 6. Bangun panel latih (sampel berimbang)

In [7]:
panel, ringkasan_panel = raw_ingest.bangun_panel_latih(
    df_suspensi_raw, symbols_universe, df_qf, df_harga, peta_sektor,
    taksonomi=taksonomi, peta_board=peta_board,
    kecualikan_tanggal=tanggal_potret, batas_akhir_as_of=batas_matang,
)
contracts.validate_panel(panel, ketat=True)
panel.to_csv(config.PATH_PANEL, index=False)

print(ringkasan_panel)
print(f"\nPanel disimpan: {config.PATH_PANEL} ({len(panel)} baris)")

if ringkasan_panel["total_kategori_c"] < 30:
    print(
        "\nPERINGATAN: kurang dari 30 peristiwa kategori C ditemukan. Model "
        "yang dilatih dari sampel sekecil ini akan punya selang kepercayaan "
        "yang lebar pada seluruh metrik (lihat docs/rancangan/metodologi.md "
        "bagian 9, poin 1). Ini bukan alasan untuk berhenti, tapi wajib "
        "disebutkan setiap kali hasil ini dilaporkan."
    )

proporsi_lengkap = panel["data_complete"].mean() if len(panel) else 0.0
print(f"\nProporsi baris panel dengan data lengkap: {proporsi_lengkap:.1%}")
if proporsi_lengkap < 0.5:
    print(
        "PERINGATAN: kurang dari separuh baris panel punya data lengkap. "
        "Notebook 03 akan menyaring baris tidak lengkap sebelum melatih -- "
        "kalau yang tersisa terlalu sedikit, notebook itu akan berhenti dengan "
        "penjelasan. Tinjau ulang cetakan bagian 3 (Diagnosa cakupan data mentah) "
        "di atas sebelum lanjut: bandingkan report_date_min/max dan "
        "harga_date_min/max dengan tanggal_potret di bagian 4, dan periksa "
        "apakah tumpang_tindih_quarterly_financials_persen serta "
        "tumpang_tindih_harga_persen mendekati 100 persen."
    )

PERINGATAN: 2 sampel positif tidak mendapat cukup kandidat negatif (kurang dari rasio 3): ['LCKM.JK', 'MGLV.JK']
{'total_suspensi_mentah': 588, 'total_kategori_c': 64, 'tak_terklasifikasi': 56, 'baris_panel': 250, 'positif_panel': 64}

Panel disimpan: D:\LIMINA\data\panel.csv (250 baris)

Proporsi baris panel dengan data lengkap: 0.0%
PERINGATAN: kurang dari separuh baris panel punya data lengkap. Notebook 03 akan menyaring baris tidak lengkap sebelum melatih -- kalau yang tersisa terlalu sedikit, notebook itu akan berhenti dengan penjelasan. Tinjau ulang cetakan bagian 3 (Diagnosa cakupan data mentah) di atas sebelum lanjut: bandingkan report_date_min/max dan harga_date_min/max dengan tanggal_potret di bagian 4, dan periksa apakah tumpang_tindih_quarterly_financials_persen serta tumpang_tindih_harga_persen mendekati 100 persen.


## 7. Pratinjau normalisasi (diagnostik, bukan scaler produksi)

Scaler yang benar-benar dipakai model dilatih ulang di notebook 03. Cell
berikut hanya untuk memeriksa sebaran fitur secara visual sebelum lanjut.

In [8]:
train_preview = splits.pisahkan_temporal(panel, cutoff_latih)
train_preview_lengkap = train_preview[train_preview["data_complete"] == 1]
X_preview = train_preview_lengkap[contracts.KOLOM_FITUR]

if len(X_preview) == 0:
    print(
        "Tidak ada baris data latih dengan data_complete == 1, jadi pratinjau "
        "standardisasi dilewati -- lihat PERINGATAN di bagian 3 dan 6 di atas "
        "untuk penyebabnya. Notebook 03 akan berhenti dengan penjelasan yang "
        "sama saat mencoba melatih."
    )
else:
    print("Ringkasan fitur SEBELUM standardisasi (data latih, baris lengkap saja):")
    display(X_preview.describe().T[["mean", "std", "min", "max"]].round(3))

    pratinjau_scaler = StandardScaler()
    X_scaled_preview = pratinjau_scaler.fit_transform(X_preview.fillna(X_preview.median()))
    print("\nSetelah standardisasi, tiap kolom seharusnya mean sekitar 0, std sekitar 1:")
    display(
        pd.DataFrame(X_scaled_preview, columns=contracts.KOLOM_FITUR)
        .describe().T[["mean", "std"]].round(3)
    )

print(
    f"\nJumlah baris latih: {len(train_preview)} total, {len(X_preview)} data lengkap, "
    f"{int(train_preview['is_event_90d'].sum())} positif"
)
print("Lanjut ke notebook 03 (pelatihan_model).")

Tidak ada baris data latih dengan data_complete == 1, jadi pratinjau standardisasi dilewati -- lihat PERINGATAN di bagian 3 dan 6 di atas untuk penyebabnya. Notebook 03 akan berhenti dengan penjelasan yang sama saat mencoba melatih.

Jumlah baris latih: 242 total, 0 data lengkap, 56 positif
Lanjut ke notebook 03 (pelatihan_model).
